# ETF行业轮动策略 - 华泰证券研报复现

**基于ETF资金流构建行业轮动策略**

本Notebook复现华泰证券金工深度研究报告《基于ETF资金流构建行业轮动策略》的核心策略逻辑。

## 1. 导入必要模块

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

sys.path.append(str(Path.cwd().parent))

from config.settings import (
    TUSHARE_TOKEN,
    TUSHARE_API_URL,
    INDUSTRY_INDEX_MAPPING,
    INDUSTRY_NAME_MAPPING,
    BACKTEST_START_DATE,
    BACKTEST_END_DATE,
)
from source.data_fetcher import TushareDataFetcher
from source.etf_flow import ETFFlowCalculator
from source.industry_rotation import IndustryRotationStrategy
from source.backtest import BacktestEngine, StrategyBacktester, plot_equity_curve
from source.utils import ensure_dir, calculate_performance_metrics

## 2. 数据获取

### 2.1 初始化数据获取器

In [ ]:
fetcher = TushareDataFetcher(token=TUSHARE_TOKEN, api_url=TUSHARE_API_URL)
print("数据获取器初始化完成")

### 2.2 获取ETF列表

In [ ]:
print("正在获取ETF列表...")
etf_df = fetcher.get_etf_basic_info()
print(f"\n共获取 {len(etf_df)} 只ETF")
if etf_df is not None and len(etf_df) > 0:
    print("\n前5只ETF:")
    display(etf_df[['ts_code', 'name', 'management']].head())

### 2.3 获取行业指数数据

In [ ]:
print("正在获取行业指数数据...")
print("（注：部分指数可能因tushare限制无法获取，需要使用其他数据源）\n")

all_index_data = []
for industry_name, index_code in INDUSTRY_INDEX_MAPPING.items():
    try:
        df = fetcher.get_index_daily(index_code, '20150101', '20240831')
        if df is not None and len(df) > 0:
            df['industry'] = industry_name
            df['index_code'] = index_code
            df['index_name'] = INDUSTRY_NAME_MAPPING.get(index_code, '')
            all_index_data.append(df)
            print(f"✓ {industry_name} ({index_code}): {len(df)} 条")
        else:
            print(f"✗ {industry_name} ({index_code}): 无数据")
    except Exception as e:
        print(f"✗ {industry_name} ({index_code}): 获取失败")

if all_index_data:
    index_returns = pd.concat(all_index_data, ignore_index=True)
    index_returns = index_returns.sort_values(['trade_date', 'industry'])
    print(f"\n成功获取 {len(index_returns)} 条指数数据")
else:
    print("\n警告: 未能获取任何指数数据")
    print("将使用模拟数据进行演示")

## 3. 生成模拟ETF资金流数据

In [ ]:
print("="*60)
print("⚠️  重要提示")
print("="*60)
print("""
Tushare不提供ETF份额日度变动数据。

实际项目中需要从以下数据源获取真实ETF申赎数据：
1. Wind金融终端
2. Choice金融终端
3. AkShare (部分数据)
4. 聚源数据

以下使用模拟数据进行策略演示。
""")

np.random.seed(42)

industry_etf_flow = {}
date_range = pd.date_range('2018-01-01', '2024-07-31', freq='D')

for industry in INDUSTRY_INDEX_MAPPING.keys():
    industry_etf_flow[industry] = pd.DataFrame({
        'trade_date': date_range,
        'net_flow': np.random.randn(len(date_range)) * 1000000,
        'nav': np.random.uniform(0.9, 1.1, len(date_range)),
        'vol': np.random.randint(1000, 100000, len(date_range)),
    })
    industry_etf_flow[industry]['trade_date'] = pd.to_datetime(
        industry_etf_flow[industry]['trade_date']
    )

print(f"已生成 {len(industry_etf_flow)} 个行业的模拟ETF资金流数据")

## 4. ETF资金流计算

In [ ]:
calculator = ETFFlowCalculator(industry_etf_flow)

print("计算周度资金流...")
weekly_flow = calculator.calculate_weekly_net_flow(
    start_date='2018-01-01',
    end_date='2024-07-31',
    method='natural_week'
)
print(f"周度资金流数据: {len(weekly_flow)} 条")
print(f"涉及行业数: {weekly_flow['industry'].nunique()}")
print(f"涉及周数: {weekly_flow[['year', 'week']].drop_duplicates().shape[0]}")

if len(weekly_flow) > 0:
    print("\n周度资金流数据预览:")
    display(weekly_flow.head(10))

### 4.1 计算滚动历史分位数

In [ ]:
print("计算滚动历史分位数 (2年窗口)...")
percentile_data = calculator.calculate_rolling_percentile(
    weekly_flow,
    window_years=2,
    percentile_col='net_flow',
    date_col='week_start',
)
print(f"滚动分位数计算完成: {len(percentile_data)} 条")

if len(percentile_data) > 0:
    print("\n分位数分布统计:")
    print(percentile_data['rolling_percentile'].describe())
    print("\n预览:")
    display(percentile_data.head(10))

## 5. 策略回测

### 5.1 策略参数设置

In [ ]:
LONG_THRESHOLD = 0.10
SHORT_THRESHOLD = 0.90
ROLLING_WINDOW_YEARS = 2

strategy = IndustryRotationStrategy(
    long_threshold=LONG_THRESHOLD,
    short_threshold=SHORT_THRESHOLD,
    rolling_window_years=ROLLING_WINDOW_YEARS,
    rebalance_freq='weekly'
)

print(f"策略: {strategy.strategy_name}")
print(f"做多阈值: {LONG_THRESHOLD} (分位数 <= {LONG_THRESHOLD})")
print(f"做空阈值: {SHORT_THRESHOLD} (分位数 >= {SHORT_THRESHOLD})")
print(f"滚动窗口: {ROLLING_WINDOW_YEARS}年")

### 5.2 生成交易信号

In [ ]:
rebalance_dates = sorted(percentile_data['week_start'].unique())
print(f"调仓日期数量: {len(rebalance_dates)}")

long_signals = strategy.generate_signals(
    percentile_data, 
    rebalance_dates, 
    signal_type='long'
)

short_signals = strategy.generate_signals(
    percentile_data, 
    rebalance_dates, 
    signal_type='short'
)

print(f"\n多头信号数量: {len(long_signals)}")
print(f"空头信号数量: {len(short_signals)}")

if len(long_signals) > 0:
    print("\n多头信号预览:")
    display(long_signals.head(10))

### 5.3 计算策略收益

In [ ]:
returns_data = index_returns.copy()
if 'trade_date' not in returns_data.columns:
    returns_data.rename(columns={'date': 'trade_date'}, inplace=True)

long_returns = strategy.calculate_portfolio_returns(long_signals, returns_data)
short_returns = strategy.calculate_portfolio_returns(short_signals, returns_data)

all_returns = pd.concat([long_returns, short_returns], ignore_index=True)

print("多头策略收益预览:")
if len(long_returns) > 0:
    display(long_returns.head(10))
else:
    print("无数据 (可能因指数数据缺失)")

## 6. 策略绩效分析

In [ ]:
metrics = strategy.calculate_performance_metrics(all_returns)

print("="*60)
print("策略绩效指标")
print("="*60)

for signal_type in ['long', 'short']:
    if signal_type not in metrics:
        continue
    
    m = metrics[signal_type]
    print(f"\n{signal_type.upper()} 策略:")
    print(f"  总收益率:     {m.get('total_return', 0):.2f}%")
    print(f"  年化收益率:   {m.get('annual_return', 0):.2f}%")
    print(f"  年化波动率:   {m.get('annual_vol', 0):.2f}%")
    print(f"  最大回撤:     {m.get('max_drawdown', 0):.2f}%")
    print(f"  夏普比率:     {m.get('sharpe', 0):.4f}")
    print(f"  卡尔玛比率:   {m.get('calmar', 0):.4f}")
    print(f"  周度胜率:     {m.get('weekly_win_rate', 0):.2f}%")
    print(f"  月度胜率:     {m.get('monthly_win_rate', 0):.2f}%")
    print(f"  盈亏比:       {m.get('profit_loss_ratio', 0):.4f}")
    print(f"  交易次数:     {m.get('num_trades', 0)}")

## 7. 可视化

### 7.1 净值曲线

In [ ]:
long_returns_df = all_returns[all_returns['signal_type'] == 'long'].copy()

if len(long_returns_df) > 0:
    long_returns_df = long_returns_df.sort_values('date')
    long_returns_df['cumulative'] = (1 + long_returns_df['mean_return'].dropna() / 100).cumprod()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(long_returns_df['date'], long_returns_df['cumulative'], 
            label=f'策略 (阈值={LONG_THRESHOLD})', linewidth=2)
    ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
    ax.set_title('ETF行业轮动策略净值曲线', fontsize=14)
    ax.set_xlabel('日期')
    ax.set_ylabel('净值')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("无足够数据进行可视化")

### 7.2 回撤曲线

In [ ]:
if len(long_returns_df) > 0:
    cumulative = long_returns_df['cumulative']
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max * 100
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.fill_between(drawdown.index, drawdown, 0, alpha=0.3, color='red')
    ax.plot(drawdown.index, drawdown, color='red', linewidth=1)
    ax.set_title('策略回撤曲线', fontsize=14)
    ax.set_xlabel('日期')
    ax.set_ylabel('回撤 (%)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("无足够数据进行可视化")

## 8. 不同阈值对比

In [ ]:
thresholds = [0.05, 0.10, 0.20]
results_comparison = []

for th in thresholds:
    test_strategy = IndustryRotationStrategy(
        long_threshold=th,
        short_threshold=1-th,
        rolling_window_years=2,
        rebalance_freq='weekly'
    )
    
    test_signals = test_strategy.generate_signals(
        percentile_data, 
        rebalance_dates, 
        signal_type='long'
    )
    
    test_returns = test_strategy.calculate_portfolio_returns(test_signals, returns_data)
    test_metrics = test_strategy.calculate_performance_metrics(test_returns)
    
    if 'long' in test_metrics:
        m = test_metrics['long']
        results_comparison.append({
            '阈值': f'{int(th*100)}%',
            '年化收益率': f"{m.get('annual_return', 0):.2f}%",
            '夏普比率': f"{m.get('sharpe', 0):.2f}",
            '最大回撤': f"{m.get('max_drawdown', 0):.2f}%",
            '卡尔玛比率': f"{m.get('calmar', 0):.2f}",
            '月度胜率': f"{m.get('monthly_win_rate', 0):.2f}%",
        })

comparison_df = pd.DataFrame(results_comparison)
print("不同阈值策略对比:")
display(comparison_df)

## 9. 结论与数据需求说明

### 9.1 研报核心结论

根据华泰证券研报《基于ETF资金流构建行业轮动策略》：

1. **核心发现**: 当行业ETF资金净流出位于历史高点时，未来短期内预期收益通常为正

2. **策略逻辑**:
   - 当ETF资金净流入历史分位数 < 5%或10%时，做多该行业
   - 当ETF资金净流入历史分位数 > 95%或90%时，做空该行业

3. **预期表现** (回测区间2018.1.1-2024.7.31):
   - 周频多头策略年化收益率 > 20%
   - Sharpe > 1
   - Calmar ≈ 1
   - 月度胜率 ≈ 70%
   - 盈亏比 > 1.4

4. **原理**: 价格压力假说 - 非信息性交易带来短期供需失衡，价格修复产生超额收益

### 9.2 数据需求

**当前状态**: Tushare不提供ETF份额日度变动数据

**需要补充的数据**:

| 数据项 | 说明 | 推荐数据源 |
|--------|------|-----------|
| ETF每日份额变动 | 用于计算资金净流入 | Wind/Choice |
| ETF净值数据 | 份额×净值=资金流入 | Wind/天天基金 |
| 真实ETF列表 | 行业ETF分类 | Wind/AkShare |

**数据计算公式**:
```
资金净流入 = 份额增加 × ETF净值
历史分位数 = rank(当期净流入 / 历史净流入) / 总期数
```

In [ ]:
print("\n" + "="*60)
print("项目说明")
print("="*60)
print("""
本项目复现了华泰证券研报的核心策略逻辑，但由于数据限制：

1. ETF资金流数据使用模拟数据
2. 指数数据可能因tushare限制部分缺失
3. 实际回测结果可能与研报有差异

要完成完整复现，请：
1. 从Wind/Choice获取真实ETF资金流数据
2. 确保21个行业指数数据完整
3. 按研报方法进行回测

项目结构:
├── config/
│   └── settings.py      # 配置文件
├── source/
│   ├── data_fetcher.py     # 数据获取
│   ├── etf_flow.py         # ETF资金流计算
│   ├── industry_rotation.py # 策略逻辑
│   ├── backtest.py         # 回测引擎
│   └── utils.py           # 工具函数
├── run.py               # 主运行脚本
└── ipynb/
    └── backtest_demo.ipynb # 本Notebook
""")